# Workflow Demonstration using Argon

Demo of the complete workflow of optimising the Lennard Jones parameters for liquid argon. For details of the different parts of the demo see the mentioned tutorials and wider MDMC documentation. This is adapted from the `Argon a-to-z` tutorial.

In [ ]:
# Imports used for this tutorial
import numpy as np
import os
from MDMC.control import Control
from MDMC.MD import Atom, Dispersion, LennardJones, Simulation, Universe

Set up the simulation box and add some Argon atoms. The mass is specified here because the experimental data set used later was for Ar36. Otherwise, the mass would be automatically look up in a reference table.
See also the `Building a Universe` tutorial for more info.

In [ ]:
# Build universe with density 0.0176 atoms per AA^-3
density = 0.0176
# This means cubic universe of side 23.0668 A will contain 216 Ar atoms
universe = Universe(dimensions=23.0668)
Ar = Atom('Ar', charge=0., mass=36.0)
# confirming the number of Ar atoms is as expected
n_ar_atoms = int(density * np.product(universe.dimensions))
print(f'Number of expected argon atoms = {n_ar_atoms}')
universe.fill(Ar, num_struc_units=(n_ar_atoms))
print(f'Number of actual argon atoms = {universe.n_atoms}')

Add an interatomic interaction between the Argon atoms using a Lennard-Jones potential. See also the `Building a Universe` and `Applying a ForceField` tutorials for more info.

In [ ]:
Ar_dispersion = Dispersion(universe,
                           (Ar.atom_type, Ar.atom_type),
                           cutoff=8.,
                           function=LennardJones(epsilon=1.0243, sigma=3.36))

In this case the interaction potential chosen is the humble Lennard Jones (to get info see doc or type `help(LennardJones)`).

Also, a `cutoff` value is chosen (see `help(Dispersion)` for more info). A [rule of thumb for Lennard-Jones](https://en.wikipedia.org/wiki/Lennard-Jones_potential) is to pick `cutoff=2.5*sigma`. The value for argon is recommended to be between 8 and 12 ang. `cutoff` is not a force-field parameter and therefore will not be refined. Ideally, and for any system you want to pick at value of the `cutoff` which is small while not compromising accuracy. For this system picking a value between 8 and 12 ang is found to give near identifical results.


Set up the MD simulation and equilibrate the system. See `Running a Simulation` tutorial for more info.

In [ ]:
# MD Engine setup
simulation = Simulation(universe,
                        engine="lammps",
                        time_step=10.18893,
                        temperature=120.,
                        traj_step=15)

In [ ]:
# Energy Minimization and equilibration
simulation.minimize(n_steps=5000)
simulation.run(n_steps=10000, equilibration=True)

Specify details about the experimental data that we want to refine against. See also the `Running a refinement` tutorial.

In [ ]:
# Dataset from: van Well et al. (1985). Physical Review A, 31(5), 3391-3414
# The resolution is approximated as 800 micro-eV FWHM based on FIG 1.(b) of the above paper.
exp_datasets = [{'file_name':'../../doc/tutorials/data/Well_s_q_omega_Ar_data.xml',
                 'type':'SQw',
                 'reader':'xml_SQw',
                 'weight':1.,
                 'auto_scale':'minimise_fom',
                 'resolution':800}]

Select the parameters to be refined and give them bounds. See also the `Selecting fitting parameters` tutorial

In [ ]:
fit_parameters = universe.parameters
fit_parameters['sigma'].constraints = [2.8,3.8]
fit_parameters['epsilon'].constraints = [0.6, 1.4]

Set up the refinement using the Gaussian-Process-Optimiser procedure and specify the length of the trajectory via `MD_steps`.

In [ ]:
control = Control(simulation=simulation,
                  exp_datasets=exp_datasets,
                  fit_parameters=fit_parameters,
                  minimizer_type="GPO",
                  reset_config=True,
                  MD_steps=12000,
                  equilibration_steps=12000)

And finally start the refinement! The parameter `n_steps` specifies the number of refinement steps to be run. It can also be specified during the previous step of creating a `Control` object.

In [ ]:
# Run the refinement, i.e. refine the FF parameters against the data
control.refine(n_steps=35)

We can now plot the output of the refinement in various ways. For example using a corner plot via a little helper function:

In [ ]:
control.plot_results();